In [10]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.3 MB/s eta 0:00:00


In [11]:
import os
import gc
import shutil
import pandas as pd

# Paths
raw_csv_path = "/content/drive/MyDrive/Colab Notebooks/Consumer-Complaint-Intelligence/data/raw/complaints.csv"
local_filtered_parquet = "/content/cfpb_filtered_raw.parquet"
drive_output_dir = "/content/drive/MyDrive/Colab Notebooks/Consumer-Complaint-Intelligence/data/processed"
drive_filtered_parquet = os.path.join(drive_output_dir, "cfpb_filtered_2021_2026.parquet")

os.makedirs(drive_output_dir, exist_ok=True)

if os.path.exists(local_filtered_parquet):
    os.remove(local_filtered_parquet)

# Read headers dynamically (skip narrative column to save GBs of RAM)
all_headers = pd.read_csv(raw_csv_path, nrows=0).columns.tolist()
columns_to_keep = [col for col in all_headers if col != "Consumer complaint narrative"]

start_date = pd.to_datetime("2021-09-26")
end_date = pd.to_datetime("2026-09-26")


In [12]:
first_chunk = True
total_filtered_rows = 0

print("Filtering raw CSV in memory-safe chunks (2021-09-26 to 2026-09-26)...")

for chunk in pd.read_csv(raw_csv_path, usecols=columns_to_keep, chunksize=250_000, low_memory=False):
    # Parse dates and filter
    chunk["Date received"] = pd.to_datetime(chunk["Date received"], errors="coerce")
    chunk = chunk[(chunk["Date received"] >= start_date) & (chunk["Date received"] <= end_date)]

    if chunk.empty:
        continue

    # Standardize column naming convention
    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace("?", "", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    # Append to local Parquet without modifying null values
    chunk.to_parquet(
        local_filtered_parquet,
        engine="fastparquet",
        append=not first_chunk,
        index=False,
        compression="snappy"
    )

    total_filtered_rows += len(chunk)
    first_chunk = False
    del chunk
    gc.collect()

print(f"Filtering complete! Rows saved: {total_filtered_rows:,}")

# Copy to Google Drive
print("Saving filtered Parquet to Google Drive...")
shutil.copyfile(local_filtered_parquet, drive_filtered_parquet)
print(f"Saved successfully at: {drive_filtered_parquet}")
print(f"File size on Drive: {round(os.path.getsize(drive_filtered_parquet) / (1024**2), 2)} MB")

Filtering raw CSV in memory-safe chunks (2021-09-26 to 2026-09-26)...
Filtering complete! Rows saved: 15,741,615
Saving filtered Parquet to Google Drive...
Saved successfully at: /content/drive/MyDrive/Colab Notebooks/Consumer-Complaint-Intelligence/data/processed/cfpb_filtered_2021_2026.parquet
File size on Drive: 497.25 MB


In [13]:
print("Reading filtered Parquet file from Google Drive...")
df = pd.read_parquet(drive_filtered_parquet)

print(f"Dataset Shape: {df.shape}")
print(f"Date range: {df['date_received'].min()} to {df['date_received'].max()}")

# Calculate missing values count and percentage
missing_audit = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

print("\n--- Missing Values Audit (Before Imputation) ---")
display(missing_audit)

Reading filtered Parquet file from Google Drive...
Dataset Shape: (15741615, 15)
Date range: 2021-09-26 00:00:00 to 2026-09-26 00:00:00

--- Missing Values Audit (Before Imputation) ---


,missing_count,missing_percent
tags,15210899,96.63
company_public_response,6774109,43.03
sub_issue,309527,1.97
state,25400,0.16
zip_code,2749,0.02
sub_product,39,0.00
date_received,0,0.00
company,0,0.00
issue,6,0.00
product,0,0.00


In [14]:
print("Handling missing values and performing feature engineering...")

# Clean whitespace on text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Meaningful missing-value replacement
df["tags"] = df["tags"].fillna("Not provided")
df["company_public_response"] = df["company_public_response"].fillna("No public response")

# Impute all remaining text columns with 'Unknown'
for col in text_cols:
    if col not in ["tags", "company_public_response"]:
        df[col] = df[col].fillna("Unknown")

# Feature engineering
df["date_sent_to_company"] = pd.to_datetime(df["date_sent_to_company"], errors="coerce")
df["year"] = df["date_received"].dt.year.astype("int16")
df["month"] = df["date_received"].dt.month.astype("int8")
df["quarter"] = df["date_received"].dt.quarter.astype("int8")
df["day_of_week"] = df["date_received"].dt.day_name()
df["response_days"] = (df["date_sent_to_company"] - df["date_received"]).dt.days.fillna(0).astype("int16")


Handling missing values and performing feature engineering...


In [15]:
df.columns.tolist()

['date_received',
 'product',
 'sub_product',
 'issue',
 'sub_issue',
 'company_public_response',
 'company',
 'state',
 'zip_code',
 'tags',
 'submitted_via',
 'date_sent_to_company',
 'company_response_to_consumer',
 'timely_response',
 'complaint_id',
 'year',
 'month',
 'quarter',
 'day_of_week',
 'response_days']

In [16]:
print("Duplicate Complaint IDs:", df["complaint_id"].duplicated().sum())

Duplicate Complaint IDs: 0


In [17]:
# Verify that missing values are now zero
print("\n--- Missing Values Audit (After Imputation) ---")
post_clean_missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "dtype": df.dtypes
})
display(post_clean_missing)

# Save final cleaned dataset to Google Drive
final_cleaned_drive_path = os.path.join(drive_output_dir, "cfpb_cleaned_2021_2026.parquet")
df.to_parquet(final_cleaned_drive_path, engine="fastparquet", index=False, compression="snappy")

print(f"\nFinal cleaned dataset saved successfully to Google Drive!")
print(f"Path: {final_cleaned_drive_path}")
print(f"Size: {round(os.path.getsize(final_cleaned_drive_path) / (1024**2), 2)} MB")


--- Missing Values Audit (After Imputation) ---


,missing_count,dtype
date_received,0,datetime64[ns]
product,0,string[python]
sub_product,0,string[python]
issue,0,string[python]
sub_issue,0,string[python]
company_public_response,0,string[python]
company,0,string[python]
state,0,string[python]
zip_code,0,string[python]
tags,0,string[python]



Final cleaned dataset saved successfully to Google Drive!
Path: /content/drive/MyDrive/Colab Notebooks/Consumer-Complaint-Intelligence/data/processed/cfpb_cleaned_2021_2026.parquet
Size: 539.07 MB
